# Does removing the Tier 0 restriction recover more ARAGORN rules?

We reran the same ten original **ChemicalEntity → inferred treats → fixed disease/phenotype**
questions against hosted **Development ARAGORN**. ARAGORN performs the expansion itself;
we do not submit eighteen manually expanded queries. The workflow is `aragorn.lookup`
(expansion, retrieval and merge; no scoring). All graph edges remain available.

Two request variants distinguish **using defaults** from **actually requesting another tier**:

| Variant | `tier` | `tiers` | `tier_fallback` |
|---|---|---|---|
| Previous Tier 0 baseline | 0 | [0] | false |
| Default rerun | omitted | omitted | omitted |
| Explicit Tier 1 rerun | 1 | [1] | false |

Retriever accepts one tier per request. Omitting tier controls defaults to Tier 0.
Fallback concerns backend availability; it does **not** automatically try another tier
when a rule is rejected or returns zero paths. Explicit Tier 1 with fallback disabled
is needed here to avoid mistaking a fallback Tier 0 answer for a Tier 1 answer.

**Running this notebook is offline.** It reads saved requests, responses and logs.
Graph release is unknown for ARAGORN's internal Retriever.


In [1]:
import json
from pathlib import Path
from collections import Counter
import pandas as pd
from IPython.display import display, Markdown

relative = Path('experiments/aragron-exploration/expansion-across-environments/followups/development-tier-selection')
candidates = [Path.cwd(), Path.cwd() / relative, Path.cwd() / 'development-tier-selection']
ROOT = next(p for p in candidates if (p / 'output/run_001').exists())
RUN = ROOT / 'output/run_001'
load = lambda p: json.loads(p.read_text())
report = load(RUN / 'report.json')
outcomes = load(RUN / 'rule-outcomes.json')
comparison = load(RUN / 'comparison.json')
config = load(RUN / 'config.json')
cases = report['cases']
display(pd.DataFrame(outcomes['environment_summary']))
print('Actual outer submissions:', len(list(RUN.glob('*/q*/submission.json'))))
print('Environment: Development only. Every request has a named submitter. No scheduled repetitions.')


Actual outer submissions: 20
Environment: Development only. Every request has a named submitter. No scheduled repetitions.


,environment,rule_evaluations,expanded,returned,rejected,empty,incomplete,error,unknown,returned_with_issues
0,default,180,180,27,80,73,0,0,0,0
1,tier1,180,0,0,0,0,0,0,180,0


## What exactly did we submit?

The only differences between the two current variants are the tier controls.
Timeout (120 seconds), DEBUG logging, bypass_cache, target IDs and workflow are held fixed.
The baseline is the earlier Development run saved in `../../output/run_002`.
These runs occurred at different times, so graph stability must not be assumed.


In [2]:
display(pd.DataFrame(config['default']['targets']))
for arm in ['default', 'tier1']:
    display(Markdown(f'**{arm}: exact first submitted request**'))
    print(json.dumps(load(RUN / arm / 'q01/request.json'), indent=2))


{
  "message": {
    "query_graph": {
      "nodes": {
        "drug": {
          "categories": [
            "biolink:ChemicalEntity"
          ]
        },
        "disease": {
          "categories": [
            "biolink:DiseaseOrPhenotypicFeature"
          ],
          "ids": [
            "MONDO:0008233"
          ]
        }
      },
      "edges": {
        "e": {
          "subject": "drug",
          "object": "disease",
          "predicates": [
            "biolink:treats"
          ],
          "knowledge_type": "inferred"
        }
      }
    }
  },
  "workflow": [
    {
      "id": "aragorn.lookup"
    }
  ],
  "parameters": {
    "timeout": 120,
    "log_level": "DEBUG",
    "bypass_cache": true
  },
  "submitter": "aragorn-development-tier-selection-2026-10-20"
}
{
  "message": {
    "query_graph": {
      "nodes": {
        "drug": {
          "categories": [
            "biolink:ChemicalEntity"
          ]
        },
        "disease": {
          "categories": [

,id,name,selection
0,MONDO:0008233,pheochromocytoma,Existing positive rule 6 example
1,MONDO:0007245,"cafe au lait spots, multiple",Existing positive rule 10 example
2,MONDO:0009665,biotinidase deficiency,Existing positive rule 11 example
3,HP:0001945,Fever,Existing positive rule 14 example
4,HP:0100812,Halitosis,Existing positive rule 17 example
5,MONDO:0100096,COVID-19,Top incoming gene-condition record count withi...
6,MONDO:0006632,"osteoarthritis, hand",Top incoming gene-condition record count withi...
7,MONDO:0007256,liver carcinoma,Next distinct target in the existing seeded 20...
8,MONDO:0009691,mycosis fungoides,Next distinct target in the existing seeded 20...
9,UMLS:C0376618,Endotoxemia,Next distinct target in the existing seeded 20...


**default: exact first submitted request**

**tier1: exact first submitted request**

## What actually executed, and what came back?

`executed_tiers` comes from **Begin processing tier-N job** log messages, not request parameters.
An empty list means no tier execution was observed. `unique_chemical_count = null` means
there was no final results list; it must not be interpreted as a successful search returning zero.

An outer HTTP 200 and `Completed` status can coexist with failed downstream lookups.
Use callback evidence to distinguish those cases. Exact answer-ID and raw-count comparisons
are withheld when either run is incomplete.


In [3]:
columns = ['environment','query','target_name','executed_tiers','received_callbacks','failed_dispatches',
           'unique_chemical_count','baseline_unique_chemicals','same_final_chemical_ids',
           'same_all_callback_counts','newly_positive_rules','complete_for_answer_comparison']
display(pd.DataFrame(comparison)[columns])


,environment,query,target_name,executed_tiers,received_callbacks,failed_dispatches,unique_chemical_count,baseline_unique_chemicals,same_final_chemical_ids,same_all_callback_counts,newly_positive_rules,complete_for_answer_comparison
0,default,q01,pheochromocytoma,[0],19,0,83.0,83,True,True,[],True
1,default,q02,"cafe au lait spots, multiple",[0],19,0,119.0,119,True,True,[],True
2,default,q03,biotinidase deficiency,[0],19,0,2.0,2,True,True,[],True
3,default,q04,Fever,[0],19,0,682.0,682,True,True,[],True
4,default,q05,Halitosis,[0],19,0,36.0,36,True,True,[],True
5,default,q06,COVID-19,[0],19,0,101.0,101,True,True,[],True
6,default,q07,"osteoarthritis, hand",[0],19,0,7.0,7,True,True,[],True
7,default,q08,liver carcinoma,[0],19,0,482.0,482,True,True,[],True
8,default,q09,mycosis fungoides,[0],19,0,28.0,28,True,True,[],True
9,default,q10,Endotoxemia,[0],19,0,2.0,2,True,True,[],True


## How is each rule identified and classified?

A callback's logged query graph is matched to the rules from the advertised ARAGORN release.
Matching preserves direction, predicates, node categories, pinned IDs and qualifier values;
only known TRAPI syntax differences and optional lookup defaults are normalized.
Rule 0 is the direct lookup and is excluded from the eighteen-rule totals.

- **Returned:** a matched rule callback has positive raw results, before ARAGORN merge filtering.
- **Rejected:** a matched callback logs an invalid predicate or unsupported typed edge pattern.
- **Empty:** one matched callback returns zero without logged failure or truncation.
- **Incomplete/error:** timeout or other failure observed in an identified callback.
- **Unknown:** insufficient evidence to identify the rule's result, including dispatch failures with no callback graph.

A zero-result rule for these ten targets is not proof of graph-wide absence.
Failed Tier 1 dispatches cannot establish rule coverage, rejection, or successful expansion of each numbered rule.


In [4]:
records = pd.DataFrame(outcomes['records'])
for arm in ['default', 'tier1']:
    display(Markdown(f'**{arm}: classification for all 18 rules and ten targets**'))
    subset = records[records.environment == arm]
    display(subset.pivot(index='rule', columns='query', values='category'))
    display(Markdown('**Raw callback counts (before merge); missing means unobserved, not zero.**'))
    display(subset.pivot(index='rule', columns='query', values='raw_result_count'))


**default: classification for all 18 rules and ten targets**

query,q01,q02,q03,q04,q05,q06,q07,q08,q09,q10
rule,,,,,,,,,,
1,rejected,rejected,rejected,rejected,rejected,rejected,rejected,rejected,rejected,rejected
2,rejected,rejected,rejected,rejected,rejected,rejected,rejected,rejected,rejected,rejected
3,rejected,rejected,rejected,rejected,rejected,rejected,rejected,rejected,rejected,rejected
4,rejected,rejected,rejected,rejected,rejected,rejected,rejected,rejected,rejected,rejected
5,rejected,rejected,rejected,rejected,rejected,rejected,rejected,rejected,rejected,rejected
6,returned,empty,empty,returned,empty,empty,empty,returned,empty,empty
7,rejected,rejected,rejected,rejected,rejected,rejected,rejected,rejected,rejected,rejected
8,rejected,rejected,rejected,rejected,rejected,rejected,rejected,rejected,rejected,rejected
9,rejected,rejected,rejected,rejected,rejected,rejected,rejected,rejected,rejected,rejected


**Raw callback counts (before merge); missing means unobserved, not zero.**

query,q01,q02,q03,q04,q05,q06,q07,q08,q09,q10
rule,,,,,,,,,,
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,411.0,0.0,0.0,9048.0,0.0,0.0,0.0,3357.0,0.0,0.0
7,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


**tier1: classification for all 18 rules and ten targets**

query,q01,q02,q03,q04,q05,q06,q07,q08,q09,q10
rule,,,,,,,,,,
1,unknown,unknown,unknown,unknown,unknown,unknown,unknown,unknown,unknown,unknown
2,unknown,unknown,unknown,unknown,unknown,unknown,unknown,unknown,unknown,unknown
3,unknown,unknown,unknown,unknown,unknown,unknown,unknown,unknown,unknown,unknown
4,unknown,unknown,unknown,unknown,unknown,unknown,unknown,unknown,unknown,unknown
5,unknown,unknown,unknown,unknown,unknown,unknown,unknown,unknown,unknown,unknown
6,unknown,unknown,unknown,unknown,unknown,unknown,unknown,unknown,unknown,unknown
7,unknown,unknown,unknown,unknown,unknown,unknown,unknown,unknown,unknown,unknown
8,unknown,unknown,unknown,unknown,unknown,unknown,unknown,unknown,unknown,unknown
9,unknown,unknown,unknown,unknown,unknown,unknown,unknown,unknown,unknown,unknown


**Raw callback counts (before merge); missing means unobserved, not zero.**

query,q01,q02,q03,q04,q05,q06,q07,q08,q09,q10
rule,,,,,,,,,,
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## What edges does each rule require?

The patterns below come from the saved advertised-release templates, not from guessed names.
For each rule, compare the default and Tier 1 outcome totals across the ten targets.
A rule rejection describes compatibility of a query pattern; it does not by itself prove an individual
predicate is absent from every record in a graph.


In [5]:
sys_path = str(ROOT.parent.parent / 'scripts')
import sys
if sys_path not in sys.path: sys.path.insert(0, sys_path)
from analyze import expected_graphs
rules = load(RUN / 'default/release-rules.json')['{"predicate": "biolink:treats"}']
graphs = expected_graphs(load(RUN / 'default/q01/request.json'), rules)
for number in range(1, 19):
    graph = graphs[number]
    display(Markdown(f'### Rule {number}'))
    edges = []
    for eid, edge in graph['edges'].items():
        edges.append({'edge': eid,
            'subject': edge['subject'] + ': ' + ', '.join(graph['nodes'][edge['subject']].get('categories', [])),
            'predicate': ', '.join(edge.get('predicates', [])),
            'object': edge['object'] + ': ' + ', '.join(graph['nodes'][edge['object']].get('categories', [])),
            'qualifiers / constraints': json.dumps(edge.get('qualifier_constraints') or edge.get('constraints') or {})})
    display(pd.DataFrame(edges))
    selected = records[records.rule == number]
    display(pd.crosstab(selected.environment, selected.category))


### Rule 1

,edge,subject,predicate,object,qualifiers / constraints
0,edge_0,g: biolink:Gene,biolink:affects_response_to,drug: biolink:ChemicalEntity,{}
1,edge_1,drug: biolink:ChemicalEntity,biolink:treats_or_applied_or_studied_to_treat,disease: biolink:DiseaseOrPhenotypicFeature,{}
2,edge_2,g: biolink:Gene,biolink:genetic_association,disease: biolink:DiseaseOrPhenotypicFeature,{}


category,rejected,unknown
environment,,
default,10,0
tier1,0,10


### Rule 2

,edge,subject,predicate,object,qualifiers / constraints
0,edge_0,drug: biolink:ChemicalEntity,biolink:treats_or_applied_or_studied_to_treat,disease: biolink:DiseaseOrPhenotypicFeature,{}
1,edge_1,h: biolink:ChemicalEntity,biolink:treats_or_applied_or_studied_to_treat,disease: biolink:DiseaseOrPhenotypicFeature,{}
2,edge_2,drug: biolink:ChemicalEntity,biolink:similar_to,h: biolink:ChemicalEntity,{}


...

,edge,subject,predicate,object,qualifiers / constraints
0,edge_0,e: biolink:ChemicalEntity,biolink:treats,disease: biolink:DiseaseOrPhenotypicFeature,{}
1,edge_1,i: biolink:BiologicalEntity,biolink:affects,drug: biolink:ChemicalEntity,"{""qualifiers"": [{""biolink:object_aspect_qualif..."
2,edge_2,i: biolink:BiologicalEntity,biolink:affects,e: biolink:ChemicalEntity,"{""qualifiers"": [{""biolink:object_aspect_qualif..."


category,empty,returned,unknown
environment,,,
default,5,5,0
tier1,0,0,10


### Rule 18

,edge,subject,predicate,object,qualifiers / constraints
0,edge_0,e: biolink:ChemicalEntity,biolink:treats_or_applied_or_studied_to_treat,disease: biolink:DiseaseOrPhenotypicFeature,{}
1,edge_1,i: biolink:BiologicalEntity,biolink:affects,drug: biolink:ChemicalEntity,"{""qualifiers"": [{""biolink:object_aspect_qualif..."
2,edge_2,i: biolink:BiologicalEntity,biolink:affects,e: biolink:ChemicalEntity,"{""qualifiers"": [{""biolink:object_aspect_qualif..."


category,empty,returned,unknown
environment,,,
default,6,4,0
tier1,0,0,10


## Read the actual evidence

These are verbatim messages from saved responses. The failed Tier 1 messages do not expose
the downstream HTTP status or response body. In the advertised worker source, a non-200
Retriever response sets `success=False` but leaves `error=None`, producing this uninformative error.
Thus “Tier 1 unavailable” is only a possible cause; the exact cause is unresolved.


In [6]:
for arm in ['default', 'tier1']:
    response = load(RUN / arm / 'q01/response.json')
    logs = response.get('logs', [])
    display(Markdown(f'**{arm}, first target: routing and failure evidence**'))
    selected = [l for l in logs if any(t in l.get('message','') for t in
                ['Sending lookup query', 'Begin processing tier', 'Failed to do lookup', 'MetaEdges could not be found', 'invalid predicates'])]
    for entry in selected[:24]:
        print(entry.get('level'), entry.get('message'))
    print('Total failed dispatches:', sum('Failed to do lookup' in l.get('message','') for l in logs))


INFO [6bb490da] Sending lookup query to http://retriever/asyncquery
INFO [d82a1d04] Sending lookup query to http://retriever/asyncquery
INFO [6bb490da] Begin processing tier-0 job d5b278189e804d1fa1ddc2e45fd5cddd for client `aragorn-development-tier-selection-2026-10-20`.
INFO [d82a1d04] Begin processing tier-0 job 040b1430578b45b3b76dc33188a407d5 for client `aragorn-development-tier-selection-2026-10-20`.
INFO [45b725bc] Sending lookup query to http://retriever/asyncquery
ERROR [6bb490da] Validation Error: Edge `edge_0` has invalid predicates: ['biolink:affects_response_to']
INFO [f1e31f18] Sending lookup query to http://retriever/asyncquery
INFO [5676d918] Sending lookup query to http://retriever/asyncquery
INFO [8e3d953e] Sending lookup query to http://retriever/asyncquery
INFO [f1e31f18] Begin processing tier-0 job 02ccc4eaf6314ae2bc7385cc9e3f5d4b for client `aragorn-development-tier-selection-2026-10-20`.
INFO [33eff5da] Sending lookup query to http://retriever/asyncquery
WARNING 

**default, first target: routing and failure evidence**

**tier1, first target: routing and failure evidence**

## Why doesn't public Retriever health resolve the failure?

ARAGORN's returned logs name **`http://retriever/asyncquery`**, an internal service URL.
The separate public endpoint **`https://dev.retriever.biothings.io/status`** reported both
Tier 0 (Gandalf) and Tier 1 (Elasticsearch) healthy at the saved preflight check.
We have not established that this public endpoint and ARAGORN's internal service are the same deployment.
Its health therefore cannot establish the state of the internal Tier 1 backend.

The evidence supports a precise conclusion: **omitting tier controls does not broaden the search**.
The explicit Tier 1 requests test the hosted ARAGORN route, but failed retrieval prevents a
coverage comparison. Additional rules may exist elsewhere; this experiment does not establish that.
The next diagnostic is to obtain the internal Retriever's rejected-submission HTTP status and body,
and verify its Tier 1 configuration, before repeating the coverage experiment.


In [7]:
health = load(ROOT / 'output/preflight/public-retriever-status.json')
print('Separate public service:', health['url'])
display(pd.DataFrame(health['body']['tiers']))
print('Reported public deployment:', health['body']['version'])
print('No graph queries are issued by this notebook.')


Separate public service: https://dev.retriever.biothings.io/status
Reported public deployment: {'git_commit': '8995c9a2f71c', 'git_branch': 'trapi-2.0'}
No graph queries are issued by this notebook.


,up,last_outage,last_recovery,error,tier,backend
0,True,None,None,None,0,gandalf
1,True,None,None,None,1,elasticsearch
